# Prioritized Variant List

This notebook produces a prioritized variant table by merging the cardiovascular association results with gnomAD variant frequency data.

In [24]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

In [25]:
associations_path = Path("../data/final/associations/association_results.csv")
gnomad_path       = Path("../data/final/gnomad/gnomad_frequencies.csv")

out_dir = Path("../data/final/prioritized_variants")
out_dir.mkdir(parents=True, exist_ok=True)

out_xlsx     = out_dir / "prioritized_variants.xlsx"
out_hf_xlsx  = out_dir / "hf_prioritized_variants.xlsx"

In [26]:
associations_df = pd.read_csv(associations_path, sep=";", low_memory=False)
gnomad_df = pd.read_csv(gnomad_path, sep=";", low_memory=False)

print(f"Association results loaded: {associations_df.shape[0]:,} rows")
print(f"gnomAD frequencies loaded:  {gnomad_df.shape[0]:,} rows")

Association results loaded: 238,651 rows
gnomAD frequencies loaded:  2,917,852 rows


In [27]:
gnomad_merge = gnomad_df[[
    "rsID",
    "gene_symbol",
    "official_gene_symbol",
    "Functional_consequence",
    "MAX_freq",
    "MAX_freq_source",
    "MAX_freq_population",
    "Priority",
]].copy()

gnomad_merge = gnomad_merge.rename(columns={
    "official_gene_symbol": "gnomad_gene_symbol",
})

gnomad_merge = gnomad_merge.dropna(subset=["rsID"]).drop_duplicates(subset=["rsID"])

In [28]:
assoc_merge = associations_df[[
    "official_gene_symbol",
    "trait_name",
    "source_dataset",
    "rsID",
    "variant_id",
    "p_value",
    "effect_size",
    "odds_ratio",
    "effect_allele",
    "other_allele",
    "allele_frequency",
    "location_relative_to_gene",
    "distance_from_gene",
    "region_assembly",
    "chromosome",
    "position",
]].copy()

assoc_merge = assoc_merge.dropna(subset=["rsID", "p_value"])

print(f"Association records with rsID and p-value available for merge: {len(assoc_merge):,}")

Association records with rsID and p-value available for merge: 233,417


In [29]:
merged_df = assoc_merge.merge(
    gnomad_merge,
    on="rsID",
    how="inner",
)

merged_df = merged_df.drop(columns=["gnomad_gene_symbol"])

print(f"Variants with both association and gnomAD data: {len(merged_df):,} rows")
print(f"Unique rsIDs in merged table: {merged_df['rsID'].nunique():,}")

Variants with both association and gnomAD data: 225,358 rows
Unique rsIDs in merged table: 49,050


In [30]:
final_columns = [
    "official_gene_symbol",
    "trait_name",
    "source_dataset",
    "rsID",
    "variant_id",
    "p_value",
    "effect_size",
    "odds_ratio",
    "effect_allele",
    "other_allele",
    "allele_frequency",
    "location_relative_to_gene",
    "distance_from_gene",
    "region_assembly",
    "chromosome",
    "position",
    "Functional_consequence",
    "MAX_freq",
    "MAX_freq_source",
    "MAX_freq_population",
    "Priority",
]

prioritized_df = merged_df[final_columns].copy()

prioritized_df = prioritized_df.rename(columns={
    "official_gene_symbol": "gene",
    "trait_name": "phenotype",
    "Functional_consequence": "functional_consequence",
    "MAX_freq_source": "gnomad_max_freq_source",
    "MAX_freq_population": "gnomad_max_freq_population",
    "MAX_freq": "gnomad_max_freq",
    "Priority": "gnomad_priority",
})

prioritized_df = prioritized_df.sort_values(
    ["gene", "p_value"],
    ascending=[True, True],
    na_position="last"
).reset_index(drop=True)

print(f"Final prioritized variant table: {prioritized_df.shape[0]:,} rows, {prioritized_df.shape[1]} columns")
prioritized_df.head(10)

Final prioritized variant table: 225,358 rows, 21 columns


,gene,phenotype,source_dataset,rsID,variant_id,p_value,effect_size,odds_ratio,effect_allele,other_allele,allele_frequency,location_relative_to_gene,distance_from_gene,region_assembly,chromosome,position,functional_consequence,gnomad_max_freq,gnomad_max_freq_source,gnomad_max_freq_population,gnomad_priority
0,APOE,Alzheimer disease; age at onset,GWAS Catalog,rs429358,NaN,0.0,NaN,NaN,?,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missense_variant,0.226612,exome,afr,Primary
1,APOE,Cholesterol:total lipids ratio; high density l...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
2,APOE,Cholesterol:total lipids ratio; high density l...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919396,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
3,APOE,Cholesteryl esters:total lipids ratio; Cholest...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,T,NaN,0.074500,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
4,APOE,Cholesteryl esters:total lipids ratio; blood V...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
5,APOE,Cholesteryl esters:total lipids ratio; blood V...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
6,APOE,Cholesteryl esters:total lipids ratio; blood V...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,T,NaN,0.075100,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
7,APOE,Cholesteryl esters:total lipids ratio; blood V...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919797,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
8,APOE,Cholesteryl esters:total lipids ratio; high de...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919396,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary
9,APOE,Cholesteryl esters:total lipids ratio; high de...,GWAS Catalog,rs7412,NaN,0.0,NaN,NaN,C,NaN,0.919344,NaN,NaN,NaN,NaN,NaN,missense_variant,0.106990,exome,afr,Primary


In [31]:
prioritized_df.to_excel(out_xlsx, index=False)

## Heart Failure Prioritized Variants

In [32]:
hf_df = prioritized_df.copy()
hf_df["phenotype"] = hf_df["phenotype"].str.strip()
hf_df["phenotype"] = hf_df["phenotype"].replace("heart failure", "Heart failure")

hf_df = hf_df[hf_df["phenotype"] == "Heart failure"].copy().reset_index(drop=True)

In [33]:
hf_df["genome_build"] = hf_df["region_assembly"]

In [38]:
hf_columns = [
    "gene",
    "phenotype",
    "source_dataset",
    "rsID",
    "variant_id",
    "chromosome",
    "position",
    "genome_build",
    "effect_allele",
    "other_allele",
    "effect_size",
    "odds_ratio",
    "p_value",
    "allele_frequency",
    "location_relative_to_gene",
    "distance_from_gene",
    "functional_consequence",
    "gnomad_max_freq",
    "gnomad_max_freq_source",
    "gnomad_max_freq_population",
    "gnomad_priority",
]

hf_df = hf_df[hf_columns].reset_index(drop=True)

print(f"HF prioritized variant table: {len(hf_df):,} rows, {hf_df.shape[1]} columns")

HF prioritized variant table: 70,856 rows, 21 columns


In [35]:
hf_df.to_excel(out_hf_xlsx, index=False)